In [ ]:
import sys      # provides system-specific parameters and functions
import re       # module for working with regular expressions (pattern matching)

# open the ORCA output file in read mode
f = open("exc30steomsingletsmallbasisset.orc", "r")

# section markers that appear in the ORCA output file
steom_results_section = "STEOM-CCSD RESULTS"
excited_state_marker = "IROOT="
steom_end_marker = "STEOM-CCSD done"
ground_state_marker = "Ground state amplitude"
absorption_spectrum_section = "ABSORPTION SPECTRUM VIA TRANSITION ELECTRIC DIPOLE MOMENTS"
cd_spectrum_section = "CD SPECTRUM"

# dictionary that will store excited states and their orbital configurations
excited_state_configurations = {}

# list placeholder (kept because it existed in the original script)
state_list=[]

# iterate through the file line by line
for line in f:

    # check if the current line contains the STEOM-CCSD results section
    if steom_results_section in line:

        # continue reading lines inside this section
        for line in f:

            # list to store configurations belonging to a single excited state
            state_configurations=[]

            # detect the start of a new excited state (IROOT)
            if excited_state_marker in line:

                # remove leading/trailing spaces and split the string at ':'
                element = line.strip().split(':')

                # normalize formatting of the state label
                element[0] = element[0].replace(" =", "=")

                # continue reading configuration lines belonging to this state
                for line in f:

                    # regex pattern extracts:
                    # 1) configuration weight
                    # 2) orbital transition (example: 100 -> 109)
                    match = re.search(r'\s+([-+]?\d*\.\d+|\d+)\s+(\d+\s*->\s*\d+)', line)

                    # if the pattern is found in the line
                    if match:

                        # first captured group = configuration weight
                        configuration_weight = match.group(1)

                        # second captured group = orbital transition
                        orbital_transition = match.group(2)

                        # append tuple (weight, transition) to the list
                        state_configurations.append((configuration_weight,orbital_transition))

                        # store the configurations under the state label
                        excited_state_configurations[element[0]] = state_configurations
                      
                    # stop collecting configurations when the ground state marker appears
                    if ground_state_marker in line:
                        #print(state_configurations)
                        break
                        
    # stop reading when the STEOM section ends
    if steom_end_marker in line:
        break

# print the dictionary containing all extracted excited state configurations
print(excited_state_configurations)

# reset the file pointer to the beginning of the file
f.seek(0)

# lists to store wavelengths and oscillator strengths
wavelengths = []
oscstrengths = []

# scan file again to locate the absorption spectrum section
for line in f:

    # detect the absorption spectrum block
    if absorption_spectrum_section in line:

        # enumerate gives both index and line while iterating
        for i, line in enumerate(f):

            # regex checks if the line starts with a numbered transition entry
            if re.search(r'^\s+[1-9]|1\d|2[0-7]$', line):

                # split the line into columns separated by whitespace
                split_line = line.split()

                # wavelength is stored in column index 2
                wavelength = split_line[2]

                # oscillator strength is stored in column index 3
                oscstrength = split_line[3]

                # append values to their respective lists
                wavelengths.append(wavelength)
                oscstrengths.append(oscstrength)

            # stop when the CD spectrum section begins
            if cd_spectrum_section in line:
                break

# print extracted spectral data
print(wavelengths)
print(oscstrengths)

# formatted table header using string formatting
print('|{:9}|{:9}||{:9}||{:9}| '.format('state', 'weight', 'transition','WAVELENGTH'))

# create a list of the excited state labels (kept from original structure)
l=list(excited_state_configurations.keys())

# set used to track states that have already been printed
seen_states = set()  # To keep track of seen states

# iterate simultaneously over states and wavelengths
for state, wavelengths in zip(excited_state_configurations.keys(), wavelengths):

    # iterate through the stored configuration lists
    for elements in excited_state_configurations.values():

        # unpack each tuple (weight, transition)
        for weight, transition in elements:

            # Check if the state is not a duplicate
            if state.strip() not in seen_states:

                # print state, weight, transition and wavelength
                print('|{:9}|{:9}||{:9}||{:9}|'.format(state.strip(), weight.strip(), transition.strip(), wavelengths.strip()))

                # mark this state as already printed
                seen_states.add(state.strip())

            else:
                # print blank state column for repeated rows
                print('|{:9}|{:9}||{:9}||{:9}|'.format('', weight.strip(), transition.strip(), wavelengths.strip()))

        # Assuming you only want the first element in `elements`
        break

{'IROOT=  1': [('0.176819', '100 -> 109'), ('-0.201811', '106 -> 109'), ('-0.940396', '108 -> 109')], 'IROOT=  2': [('0.282472', '100 -> 109'), ('-0.899440', '106 -> 109'), ('0.279872', '108 -> 109')], 'IROOT=  3': [('0.211522', '99 -> 109'), ('-0.360846', '101 -> 109'), ('-0.885692', '107 -> 109')], 'IROOT=  4': [('0.123051', '103 -> 109'), ('0.972520', '105 -> 109'), ('0.113186', '105 -> 118')], 'IROOT=  5': [('-0.149660', '103 -> 111'), ('0.692849', '104 -> 110'), ('-0.665608', '105 -> 111'), ('0.105401', '105 -> 114'), ('-0.124905', '108 -> 110')], 'IROOT=  6': [('-0.127257', '99 -> 109'), ('-0.886924', '101 -> 109'), ('0.345887', '107 -> 109'), ('-0.136464', '108 -> 114')], 'IROOT=  7': [('0.127778', '99 -> 109'), ('0.889704', '101 -> 109'), ('-0.343551', '107 -> 109'), ('0.131327', '108 -> 114')], 'IROOT=  8': [('0.499561', '106 -> 112'), ('0.562024', '107 -> 113'), ('0.637230', '108 -> 112')], 'IROOT=  9': [('0.494024', '106 -> 113'), ('0.614050', '107 -> 112'), ('0.591215', '10